# Compass Th17

This notebook follows the whole analysis on Compass scores, revisiting the analysis of Th17 cells in [Wagner et al., Cell 2021](https://doi.org/10.1016/j.cell.2021.05.045). Its dataset, Compass_Th17, holds 151 non-pathogenic (Th17n) and 139 pathogenic (Th17p) samples, and the question is which metabolic reactions and pathways differ between the two. The steps are those of the Metabis server's Compass Th17 tutorial, made with the library calls the server makes, so the numbers are the app's.

## Before you run it

The notebook is self-contained. Install the repository with its development dependencies, then download the example data from the `metabis` directory:

```bash
python data/download_datasets.py
```

The notebook reads the three Th17 Compass files from `metabis/data`.

In [ ]:
# ruff: noqa: E402, I001
import os
import sys
import tempfile
import warnings
from pathlib import Path

import pandas as pd
from bokeh.io import output_notebook, show

os.environ.setdefault(
    "MPLCONFIGDIR",
    str(Path(tempfile.gettempdir()) / "metabis-matplotlib"),
)
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore", message="IProgress not found.*")


def find_metabis_repo_root_for_import(start: Path) -> Path:
    direct_candidates = [start, *start.parents]
    nested_candidates = [candidate / "metabis" for candidate in direct_candidates]
    for candidate in [*direct_candidates, *nested_candidates]:
        if (candidate / "src" / "metabis").exists():
            return candidate
    raise FileNotFoundError("Could not locate the Metabis repository root.")


repo_root_for_import = find_metabis_repo_root_for_import(Path.cwd().resolve())
src_path = repo_root_for_import / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from metabis import (
    DEFAULT_LANDSCAPE_DELTA,
    ActivityConfig,
    ActivityOverlay,
    AdjustmentColumn,
    AnalysisWorkspace,
    ComparisonAdjustment,
    ExperimentAnalysis,
    ExpressionData,
    ExpressionPreprocessingConfig,
    GroupComparison,
    GroupComparisonSpec,
    GroupSummary,
    MetabolicModel,
    ReactionData,
    ReactionPreprocessingConfig,
    SampleMetadata,
    add_category_key,
    add_map_bands,
    add_pathway_key,
    band_name,
    biplot_arrow_features,
    build_reduction_presentation,
    category_numbers,
    element_details,
    fit_view_at_one_scale,
    highlight_reactions,
    keep_map_scale,
    map_bands,
    map_ids_for_reactions,
    maps_for_reactions,
    overview_color_bar,
    plot_pca_components_artifact,
    plot_reduction_artifact,
    plot_sample_heatmap,
    plot_set_enrichment_category,
    plot_statistical_scatterplots,
    plot_subsystem_activity_ranking,
    plot_subsystem_comparison,
    sample_heatmap_block,
    subsystem_delta,
    subsystem_delta_ranking,
    subsystem_members,
    with_thresholds,
)
from metabis.analysis.set_enrichment import (
    SetEnrichmentConfig,
    SetEnrichmentFeatureType,
    build_set_enrichment_annotations,
    run_comparison_set_enrichment,
)
from metabis.analysis_workspace import RevisionKind
from metabis.version import __version__ as metabis_version
from metabis.viz.metabolic_visualization import (
    DEFAULT_SUBSYSTEM_LAYOUT_ENGINE,
    EnrichmentConfig,
    GraphCalculationInput,
    GraphGenerationConfig,
    MetabolicGraphPipeline,
    PipelineConfig,
    RenderingConfig,
    render_graph_artifact,
)
from metabis.viz.plot_landscape import sample_heatmap_rows

output_notebook()

In [ ]:
def find_metabis_repo_root(start: Path) -> Path:
    direct_candidates = [start, *start.parents]
    nested_candidates = [candidate / "metabis" for candidate in direct_candidates]
    for candidate in [*direct_candidates, *nested_candidates]:
        if (
            (candidate / "pyproject.toml").exists()
            and (candidate / "demo").exists()
            and (candidate / "src" / "metabis").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the Metabis repository root.")


repo_root = find_metabis_repo_root(Path.cwd().resolve())
data_dir = repo_root / "data"
required_example_files = (
    "Th17_Compass_reactions.tsv",
    "Th17_Compass_expression.tsv",
    "Th17_Compass_metadata.tsv",
)
missing = [name for name in required_example_files if not (data_dir / name).is_file()]
if missing:
    raise FileNotFoundError(
        "Missing example files: "
        + ", ".join(missing)
        + ". Run `python data/download_datasets.py`."
    )

reaction_data_path = data_dir / "Th17_Compass_reactions.tsv"
expression_data_path = data_dir / "Th17_Compass_expression.tsv"
metadata_path = data_dir / "Th17_Compass_metadata.tsv"

print(f"Metabis version: {metabis_version}")
print(f"Reaction data: {reaction_data_path}")
print(f"Expression data: {expression_data_path}")
print(f"Sample metadata: {metadata_path}")

## 1. Load the data

`MetabolicModel` is the packaged Human1 model, on which every analysis runs. Compass scored the reactions of RECON2: `get_source_reactions("RECON2")` gives RECON2's reaction table, which maps each of its reactions to the Human1 reaction it corresponds to.

In [ ]:
model = MetabolicModel(use_cache=True)
source_reactions = model.get_source_reactions("RECON2")

print(
    f"Human1: {len(model.get_reactions()):,} reactions, {len(model.get_genes()):,} genes, "
    f"{len(model.get_subsystems())} subsystems"
)
print(f"RECON2 reaction table: {len(source_reactions):,} reactions")

`ReactionData.create_with_validation` reads the reaction matrix and maps its reactions onto the model. Compass writes each reaction's two directions as a `_pos` and a `_neg` column, read in dual mode. It gives each reaction a penalty, low where the reaction is likely active, and `compass_activity=True` turns the penalties into activity scores; `zero_below=1e-6` is the server's default. The Human1 reactions RECON2 writes the other way round are then reversed in the model, as the server reverses them.

In [ ]:
reaction_data, reaction_validation = ReactionData.create_with_validation(
    str(reaction_data_path),
    "dual",
    metabolic_model=model,
    reaction_name_format="rxnRecon3DID",
    csv_separator="\t",
    samples_in="columns",
    check_old_identifiers=True,
    preprocessing=ReactionPreprocessingConfig(compass_activity=True, zero_below=1e-6),
    source_reactions=source_reactions,
)
assert reaction_validation["status"] == "success", reaction_validation["messages"]
model.reverse_reactions(sorted(reaction_data.reversed_reactions))

alignment = reaction_validation["alignment_report"]
blocked = reaction_validation["preprocessing_report"].parameters["blocked_reactions"]
n_samples, n_reactions = reaction_data.get_reaction_df().shape
subsystems = (alignment["subsystems_with_reactions"], alignment["total_subsystems"])
print(f"Samples: {n_samples}")
print(f"Reactions: {n_reactions:,} loaded")
print(
    f"Human1 coverage: {alignment['model_coverage_percent']:.1f}% "
    f"({alignment['base_reactions_mapped']:,} of {alignment['total_model_reactions']:,} reactions)"
)
print(f"Subsystems: {subsystems[0]} of {subsystems[1]} covered")
print(f"Not mapped: {alignment['unmapped_count']:,} reactions")
print(f"Blocked reactions: {blocked:,} removed")

assert (n_samples, n_reactions) == (290, 5915)
assert f"{alignment['model_coverage_percent']:.1f}" == "36.9"
assert subsystems == (114, 154)
assert blocked == 3285

These are the key figures the app lists after the load: 290 samples and 5,915 reactions, which cover 36.9% of Human1 and 114 of its 154 subsystems, with 3,285 blocked reactions removed. Blocked reactions are those Compass found unable to carry flux in any sample; they hold no information, so they are removed.

`ExpressionData.create_with_validation` maps the expression matrix's gene symbols onto the model's genes and keeps the metabolic ones. The example's expression is normalized for library size but on a linear scale, so it is loaded with log1p.

In [ ]:
expression_data, expression_validation = ExpressionData.create_with_validation(
    str(expression_data_path),
    gene_list=model.get_genes(),
    metabolic_model=model,
    gene_format="geneSymbols",
    preprocessing=ExpressionPreprocessingConfig(log1p=True),
    data_format="csv",
    csv_separator="\t",
    samples_in="columns",
)
assert expression_validation["status"] == "success", expression_validation["messages"]

gene_alignment = expression_validation["alignment_report"]
print(
    f"Genes: {gene_alignment['mapped_count']:,} of the model's {gene_alignment['total_model_genes']:,} "
    f"metabolic genes ({gene_alignment['model_coverage_percent']:.1f}%), in "
    f"{expression_data.get_expression_df().shape[0]} samples"
)

`SampleMetadata` reads the sample sheet, whose `sample` column holds the reaction matrix's sample IDs. `AnalysisWorkspace` holds the model, both matrices and the sheet: every later step reads from it, and its results are installed in it.

In [ ]:
metadata = SampleMetadata.from_table(metadata_path, sample_id_column="sample")
dm = AnalysisWorkspace(
    metabolic_model=model,
    reaction_data=reaction_data,
    expression_data=expression_data,
)
metadata_alignment = dm.update_sample_metadata(metadata)

print(f"Sheet columns: {list(metadata.columns)}")
print(f"Aligned samples: {len(metadata_alignment.aligned_samples)}")
assert (
    len(metadata_alignment.aligned_samples) == 290
    and metadata_alignment.is_strict_match
)

The sheet describes all 290 samples.

## 2. Groups and the samples at a glance

Groups are the sets of samples every later step works with. `load_groups_from_metadata` makes one group per value of a sheet column, here `cell_type`.

In [ ]:
dm.load_groups_from_metadata(["cell_type"])
group_sizes = {group: len(dm.get_group_samples(group)) for group in dm.get_group_ids()}
print(group_sizes)
assert group_sizes == {"Th17p": 139, "Th17n": 151}

`ExperimentAnalysis` summarizes each group, which the single-group activity of section 4 reads, and fits the reductions: the app's map is a PCA of the 5,000 most variable features. Each result is installed in the workspace with the state of the workspace it was computed from, which `capture_analysis_snapshot` records.

In [ ]:
analysis_dependencies = frozenset(
    {
        RevisionKind.DATASET,
        RevisionKind.METADATA,
        RevisionKind.GROUPS,
        RevisionKind.PREPROCESSING,
    }
)
analysis_snapshot = dm.capture_analysis_snapshot(analysis_dependencies)

analysis = ExperimentAnalysis(dm, auto_run=False)
outcome = analysis.run_analysis(
    include_pca=True,
    pca_data_types=["reactions", "expression"],
    n_variable_features=5000,
)
assert outcome["status"] == "success", outcome["problems"]
for result in analysis.get_reduction_results():
    analysis.install_reduction_artifact(
        result.build_artifact(
            provenance=analysis_snapshot.provenance,
            source_fingerprints=dict(dm.source_fingerprints),
        )
    )
dm.install_experiment_analysis(analysis_snapshot.provenance, analysis)
dm.refresh_experiment_analysis_presentation()

reaction_pca = dm.get_reduction_artifact("PCA", "reactions")
expression_pca = dm.get_reduction_artifact("PCA", "expression")
print(
    f"Reaction PCA: the {reaction_pca.n_variable_features_used:,} most variable of {n_reactions:,} reactions"
)
print(f"Expression PCA: all {expression_pca.n_variable_features_used:,} genes")
assert reaction_pca.n_variable_features_used == 5000

`build_reduction_presentation` freezes the labels, colors and arrows a reduction figure reads, and `plot_reduction_artifact` draws it: each point is a sample, colored by its group. The biplot arrows, on in the app from the start, draw up to 8 features, the strongest in each direction, each pointing to where the samples that score it highest lie. The line under the plot rates how well the groups separate.

In [ ]:
def reduction_figure(artifact, **options):
    presentation = build_reduction_presentation(
        dm,
        artifact,
        width=650,
        height=480,
        show_biplot_arrows=True,
        biplot_top_arrows=8,
        biplot_arrow_scale=1.5,
        **options,
    )
    return plot_reduction_artifact(artifact, presentation)


show(reduction_figure(reaction_pca))

`plot_sample_heatmap` draws the 45 most variable reactions, each scaled across the samples, gold above its average, light gray at it and purple below, the samples ordered by group under a band in its color.

In [ ]:
sample_groups = {
    str(sample): group
    for group in sorted(dm.get_group_ids())
    for sample in dm.get_group_samples(group)
}
reaction_scores = reaction_data.get_reaction_df()
reaction_names = {
    reaction: model.get_reaction_reference_data(reaction).get("short_name") or reaction
    for reaction in reaction_scores.columns
}
show(
    plot_sample_heatmap(
        reaction_scores,
        sample_groups,
        count=45,
        labels=reaction_names,
        group_colors=dm.get_group_colors(),
        title="45 most variable reactions",
        width=600,
        data_type="reactions",
        label_limit=30,
    )
)

The heatmap's first rows, with each group's mean score:

In [ ]:
group_of_sample = pd.Series(sample_groups)
group_means = reaction_scores.loc[group_of_sample.index].groupby(group_of_sample).mean()
heatmap_rows = sample_heatmap_rows(reaction_scores.loc[group_of_sample.index], 45)
top_rows = pd.DataFrame(
    [
        {
            "reaction": row[0],
            "subsystem": ", ".join(
                model.get_reaction_metadata(row[0].removesuffix("_back"))["subsystems"]
            ),
            "Th17n mean": group_means.at["Th17n", row[0]],
            "Th17p mean": group_means.at["Th17p", row[0]],
        }
        for row in heatmap_rows[:12]
    ]
).set_index("reaction")
mitochondrial_transport = top_rows[
    top_rows["subsystem"].str.contains("Transport, mitochondrial")
]
assert len(mitochondrial_transport) >= 3
assert (
    mitochondrial_transport["Th17n mean"] > mitochondrial_transport["Th17p mean"]
).all()
top_rows.round(3)

The reaction PCA mixes Th17p and Th17n, as the line under it rates. A few rows of the heatmap shift between the groups: the mitochondrial transport rows near the top run higher in Th17n than in Th17p. Plotting the expression instead shows the groups apart.

In [ ]:
show(reduction_figure(expression_pca))

gene_names = {
    gene: model.get_gene_reference_data(gene).get("display_label") or gene
    for gene in expression_data.get_expression_df().columns
}
show(
    plot_sample_heatmap(
        sample_heatmap_block(expression_data.matrix(), list(sample_groups), 45),
        sample_groups,
        count=45,
        labels=gene_names,
        group_colors=dm.get_group_colors(),
        title="45 most variable genes",
        width=600,
        data_type="expression",
        label_limit=30,
    )
)

The expression PCA's first component separates Th17p from Th17n. `plot_pca_components_artifact` gives each component's share of the variance, in the bars, over the loadings of the genes that weigh most in them: teal where a gene rises along a component, brown where it falls. The two components the map plots are outlined, and the genes it draws as arrows, which `biplot_arrow_features` names, are marked.

In [ ]:
arrow_genes = biplot_arrow_features(expression_pca, 1, 2, 8)
components = build_reduction_presentation(
    dm,
    expression_pca,
    width=650,
    height=480,
    loading_component_count=8,
    loading_feature_count=14,
    loading_sort_component=None,
)
show(
    plot_pca_components_artifact(
        expression_pca, components, plotted=(1, 2), arrow_features=arrow_genes
    )
)

In [ ]:
expression = expression_data.get_expression_df()
pc1_share = expression_pca.pca.explained_variance_ratio[0]
pc1_loadings = pd.Series(
    expression_pca.pca.loadings[0], index=list(expression_pca.feature_names)
)
strongest = pc1_loadings.abs().idxmax()
pc1_scores = pd.Series(
    expression_pca.coordinates[:, 0], index=list(expression_pca.sample_ids)
)
pc1_means = pc1_scores.groupby(group_of_sample).mean()
aqp3 = model.translate_gene(
    "AQP3", input_format="geneSymbols", output_format="ensmbl_id"
)
aqp3_means = expression[aqp3].groupby(group_of_sample).mean()

print(f"PC1 explains {pc1_share:.1%} of the variance in expression")
print(
    f"Mean PC1 score: Th17n {pc1_means['Th17n']:+.2f}, Th17p {pc1_means['Th17p']:+.2f}"
)
print(f"Largest PC1 loading: {gene_names[strongest]} ({pc1_loadings[strongest]:+.3f})")
print(f"Arrows, longest first: {', '.join(gene_names[gene] for gene in arrow_genes)}")
print(
    f"AQP3, mean log1p expression: Th17n {aqp3_means['Th17n']:.2f}, Th17p {aqp3_means['Th17p']:.2f}"
)

assert f"{pc1_share:.1%}" == "2.5%"
assert strongest == aqp3 and arrow_genes[0] == aqp3
assert (pc1_loadings[aqp3] > 0) == (pc1_means["Th17n"] > 0) != (pc1_means["Th17p"] > 0)
assert aqp3_means["Th17n"] > aqp3_means["Th17p"]

PC1, the axis that sets Th17p apart from Th17n, explains 2.5% of the variance in expression. AQP3 has its largest loading and the longest arrow, which points to the Th17n samples, and it is higher in them. `colour_by` colors the samples by one reaction or gene instead, gold above its average and purple below; by AQP3:

In [ ]:
show(reduction_figure(expression_pca, colour_by=("expression", aqp3)))

And by SAT1:

In [ ]:
sat1 = model.translate_gene(
    "SAT1", input_format="geneSymbols", output_format="ensmbl_id"
)
show(reduction_figure(expression_pca, colour_by=("expression", sat1)))

sat1_means = expression[sat1].groupby(group_of_sample).mean()
print(
    f"SAT1, mean log1p expression: Th17n {sat1_means['Th17n']:.2f}, Th17p {sat1_means['Th17p']:.2f}"
)
assert sat1_means["Th17p"] > sat1_means["Th17n"]

The samples on the Th17p side of the first component run gold and those on the Th17n side gray to purple: in these data *SAT1*, the spermidine/spermine acetyltransferase of polyamine turnover, is more expressed in the pathogenic cells. Wagner et al. measured it higher in Th17p than in Th17n by qPCR (Fig. 4D), and section 3 tests it here.

## 3. Compare Th17p with Th17n

`GroupComparison` tests every reaction, and every gene, for a difference between two sides. `GroupComparisonSpec.pairwise` names them: Th17p, the Target whose changes are reported, against Th17n, the Baseline. Every signed result is Th17p against Th17n: positive means higher in Th17p, negative higher in Th17n. The test is the Wilcoxon rank-sum test, its p-values corrected with Benjamini-Hochberg (FDR), the app's defaults.

`build_artifact` freezes the results, and the workspace installs them. `apply_comparison_thresholds` then decides which features count as changed: here those below an adjusted p of 0.1, for reactions and for genes, as the tutorial sets both.

In [ ]:
comparison_spec = GroupComparisonSpec.pairwise(
    ("Th17p",),
    ("Th17n",),
    target_label="Th17p",
    baseline_label="Th17n",
)
comparison = GroupComparison(spec=comparison_spec, test="wilcoxon", correction="fdr_bh")
for for_genes in (False, True):
    comparison.run_test(dm, for_genes=for_genes, test="wilcoxon", correction="fdr_bh")

comparison_snapshot = dm.capture_analysis_snapshot(analysis_dependencies)
dm.install_group_comparison_artifact(
    comparison_snapshot.provenance,
    comparison.build_artifact(
        provenance=comparison_snapshot.provenance,
        source_fingerprints=dict(dm.source_fingerprints),
    ),
)
dm.refresh_group_comparison_presentation()

thresholds = {
    "pval_threshold": 0.1,
    "cohens_d_threshold": 0.0,
    "log2_fc_threshold": 0.0,
    "mean_difference_threshold": 0.0,
}
significant_reactions, reaction_report = dm.apply_comparison_thresholds(
    **thresholds, for_genes=False
)
significant_genes, gene_report = dm.apply_comparison_thresholds(
    **thresholds, for_genes=True
)

judged = dm.get_group_comparison_artifact()
reaction_results = judged.get_test_results(for_genes=False)
gene_results = judged.get_test_results(for_genes=True)
print(
    f"Reactions: {len(significant_reactions):,} of {reaction_report['total_elements']:,} "
    f"significant at adjusted p < 0.1"
)
print(
    f"Genes: {len(significant_genes):,} of {gene_report['total_elements']:,} significant at adjusted p < 0.1"
)
assert (len(significant_reactions), reaction_report["total_elements"]) == (3039, 5915)

In the Target, Th17p, 3,039 of the 5,915 reactions are significant.

The analysis also summarizes each subsystem. The Delta score is Metabis's own summary, not a standard statistic. With the app's settings, printed below, it is the sum of the signs of the significant reactions' Cohen's d over the subsystem's measured reactions, from −1 (every reaction significantly higher in Th17n) to +1 (every one higher in Th17p). A reversible reaction counts once, in the direction the reaction direction criterion chooses. `subsystem_delta` scores every subsystem, and `subsystem_members` lists the reactions each one counts.

In [ ]:
delta_config = {
    **DEFAULT_LANDSCAPE_DELTA,
    "reaction_direction_criterion": dm.reaction_direction_criterion.value,
}
print(delta_config)

subsystem_scores = subsystem_delta(
    model, reaction_data, judged, delta_config=delta_config
)
members = subsystem_members(model, judged, criterion=dm.reaction_direction_criterion)


def subsystem_counts(subsystem_id):
    """A subsystem's measured and significant reactions, by direction, and its Delta score."""
    results = reaction_results.loc[list(members[subsystem_id])]
    significant = results[results["is_significant"]]
    return {
        "measured": len(results),
        "significant": len(significant),
        "higher in Th17p": int((significant["cohens_d"] > 0).sum()),
        "higher in Th17n": int((significant["cohens_d"] < 0).sum()),
        "Delta score": round(float(subsystem_scores.at[subsystem_id, "delta"]), 3),
    }


worked = pd.DataFrame.from_dict(
    {
        subsystem_scores.at[subsystem_id, "name"]: subsystem_counts(subsystem_id)
        for subsystem_id in (
            "glycolysis_gluconeogenesis",
            "vitamin_b2_metabolism",
            "thiamine_metabolism",
        )
    },
    orient="index",
)
assert worked.loc["Glycolysis / Gluconeogenesis"].tolist() == [32, 23, 21, 2, 0.594]
assert worked["measured"].tolist() == [32, 1, 6]
assert worked["Delta score"].tolist() == [0.594, 1.0, 1.0]
worked

Glycolysis / Gluconeogenesis has 32 measured reactions. At adjusted p below 0.1, 23 of them differ, 21 higher in Th17p and 2 higher in Th17n, so its Delta score is (21 − 2) / 32 = +0.59. A subsystem whose few reactions all move one way scores ±1 however small it is: Vitamin B2 metabolism, one reaction, scores +1, and so does Thiamine metabolism, with six. The score describes and has no p-value; the over-representation analysis of section 4 tests.

`plot_statistical_scatterplots` draws the volcano plot: each reaction's Cohen's d against its adjusted p-value, colored where significant, each side named at its own end.

In [ ]:
show(
    plot_statistical_scatterplots(
        dm,
        return_plot=True,
        show_thresholds=True,
        for_genes=False,
        merge_tools=False,
        volcano="cohens_d",
        volcano_y="adjusted",
        sides=("Th17n", "Th17p"),
    )
)

Arginine and proline metabolism holds the polyamine pathway of *SAT1*, colored in section 2. Its reactions, as the app's results table lists them once the subsystem is kept, the most significant first:

In [ ]:
arginine_counts = subsystem_counts("arginine_and_proline_metabolism")
print(arginine_counts)
assert [
    arginine_counts[key]
    for key in ("measured", "significant", "higher in Th17p", "higher in Th17n")
] == [49, 34, 19, 15]

arginine_results = reaction_results.loc[
    list(members["arginine_and_proline_metabolism"])
]
arginine_results.assign(
    name=[
        model.get_reaction_metadata(reaction.removesuffix("_back"))["name"]
        for reaction in arginine_results.index
    ]
)[
    ["name", "adjusted_pval", "cohens_d", "mean_difference", "is_significant"]
].sort_values("adjusted_pval").head(10)

Its reactions sit on both sides of zero: 34 of its 49 measured reactions differ, 19 higher in Th17p and 15 higher in Th17n, so its Delta score stays near zero though most of them change.

## 4. Pathways

Human1 sorts its reactions into subsystems, such as glycolysis. This section summarizes the comparison per subsystem, draws all of them as one network, and opens any of them as a pathway graph. `MetabolicGraphPipeline.build_artifact` builds a graph as plain data, colored from what the workspace holds, and `render_graph_artifact` draws it. The functions below build and draw them with the server's settings: the network over its color key and the key of its numbered categories, and a subsystem's graph on its Human1 map layout over the key of its shapes and colors (`add_pathway_key`).

In [ ]:
HOVER_DETAIL = {
    "full": dict(
        show_hover_stats=True,
        show_hover_distribution_plots=True,
        show_hover_color_reference=True,
    ),
    "medium": dict(
        show_hover_stats=True,
        show_hover_distribution_plots=False,
        show_hover_color_reference=True,
    ),
    "light": dict(
        show_hover_stats=True,
        show_hover_distribution_plots=False,
        show_hover_color_reference=False,
    ),
}


def build_graph(element, graph_type, enrichment, rendering, source):
    pipeline = MetabolicGraphPipeline(
        model,
        PipelineConfig(
            generation_config=GraphGenerationConfig(
                use_human1_map=True,
                layout_engine="fdp"
                if graph_type == "metabolic_network"
                else DEFAULT_SUBSYSTEM_LAYOUT_ENGINE,
                resource_registry=model.resource_registry,
            ),
            enrichment_config=enrichment,
            show_plot=False,
        ),
    )
    snapshot = dm.capture_analysis_snapshot(analysis_dependencies)
    artifact = pipeline.build_artifact(
        element,
        graph_type,
        data_manager=source,
        provenance=snapshot.provenance,
        source_fingerprints=dict(dm.source_fingerprints),
    )
    return artifact, render_graph_artifact(artifact, rendering).plot


def draw_network(color_metric, *, delta_options=None, source=None, **color_bar_options):
    """The metabolic network, as the app's overview draws it, over its color key."""
    _, plot = build_graph(
        None,
        "metabolic_network",
        EnrichmentConfig(
            color_metric=color_metric,
            delta_options=delta_options,
            subsystem_family="reaction",
            **HOVER_DETAIL["full"],
        ),
        # Room for the network and, below it, its color key and the category key.
        RenderingConfig(
            width=975,
            height=950,
            sizing_mode="stretch_width",
            aspect_ratio=None,
            zoom=1.5,
            match_aspect=True,
        ),
        source or dm,
    )
    color_bar = overview_color_bar(dm, delta_options=delta_options, **color_bar_options)
    color_bar.update(orientation="horizontal", width=200, height=10, location="center")
    plot.add_layout(color_bar, "below")
    add_category_key(plot, visible=True)
    # At one scale in whatever room the keys leave, as the app's overview.
    return fit_view_at_one_scale(plot)


def draw_pathway(element, graph_type="subsystem", hover="medium"):
    """A subsystem's graph, or a compartment's map, colored by the comparison."""
    artifact, plot = build_graph(
        element,
        graph_type,
        EnrichmentConfig(
            color_metric="cohens_d", delta_options=delta_config, **HOVER_DETAIL[hover]
        ),
        RenderingConfig(
            width=975,
            height=900,
            sizing_mode="stretch_width",
            aspect_ratio=None,
            zoom=1.0,
            match_aspect=True,
            fit_view_to_bounds=True,
        ),
        dm,
    )
    add_pathway_key(
        plot, dm, visualization_mode="group_comparison", color_metric="cohens_d"
    )
    # Its view whole and at one scale above its key, as the app draws it, framed
    # or not (highlight_reactions): fitted again as the notebook settles its width.
    keep_map_scale(plot, fit_whole=True)
    return artifact, plot

### Summarize subsystems with Delta score

The Delta score colors each subsystem, from blue, when every reaction is significantly higher in Th17n, to red, when every one is higher in Th17p. The network draws one circle per subsystem, its area following its reactions and its color its score, inside its numbered category; a white circle with a dashed edge has no measured reaction. Lines join subsystems that share metabolites.

In [ ]:
show(
    draw_network(
        "cohens_d", delta_options=delta_config, visualization_mode="group_comparison"
    )
)

`subsystem_delta_ranking` ranks the subsystems as the app's ranking beside the network does, and `plot_subsystem_comparison` draws it: each subsystem's Delta score, under its category, beside each of its reactions' Cohen's d as a dot.

In [ ]:
delta_ranking = subsystem_delta_ranking(
    model, reaction_data, judged, delta_config=delta_config
)
show(
    plot_subsystem_comparison(
        dm,
        delta_ranking,
        metric="cohens_d",
        max_points_per_subsystem=50,
        return_plot=True,
        delta_options=delta_config,
    )
)

### Read one category

In the app, a click on a category's number zooms the network to it and names its subsystems. Central Carbon & Bioenergetics is number 1; its subsystems and their scores:

In [ ]:
numbers = category_numbers(
    category
    for category, subsystems in model.get_subsystems_by_category().items()
    if subsystems
)
central_carbon = model.get_subsystems_by_category()["Central Carbon & Bioenergetics"]
print(
    f"Category {numbers['Central Carbon & Bioenergetics']}: Central Carbon & Bioenergetics"
)
assert numbers["Central Carbon & Bioenergetics"] == 1

category_table = pd.DataFrame.from_dict(
    {
        subsystem_scores.at[subsystem_id, "name"]: subsystem_counts(subsystem_id)
        for subsystem_id in central_carbon
    },
    orient="index",
).sort_values("Delta score", ascending=False)
expected = {
    "Oxidative phosphorylation": (0.67, 6, 9),
    "Pentose phosphate pathway": (0.62, 13, 21),
    "Glycolysis / Gluconeogenesis": (0.59, 23, 32),
    "Pyruvate metabolism": (0.56, 14, 25),
    "Tricarboxylic acid cycle and glyoxylate/dicarboxylate metabolism": (0.33, 14, 30),
}
assert {
    name: (round(row["Delta score"], 2), row["significant"], row["measured"])
    for name, row in category_table.iterrows()
} == expected
category_table

All five of its subsystems are red, higher in Th17p: Oxidative phosphorylation (+0.67, 6 of its 9 measured reactions differ), the Pentose phosphate pathway (+0.62, 13 of 21), Glycolysis / Gluconeogenesis (+0.59, 23 of 32), Pyruvate metabolism (+0.56, 14 of 25) and the TCA cycle with glyoxylate and dicarboxylate metabolism (+0.33, 14 of 30). Wagner et al. predicted that "glycolytic reactions were generally more active in Th17p than in Th17n", and increased activity in Th17p "through two segments of the TCA cycle, but not the cycle as a whole" (Figs. 2C, 3A). A score counts a subsystem's changes; its graph shows where they are.

### Inspect one pathway

Glycolysis / Gluconeogenesis on its Human1 layout, the curated Metabolic Atlas map: reactions as diamonds, colored by Cohen's d where significant and gray when measured but not, genes as rectangles colored by their own test, white with a dashed edge where nothing was measured, and metabolites as ovals.

In [ ]:
glycolysis_artifact, glycolysis_graph = draw_pathway("Glycolysis / Gluconeogenesis")
show(glycolysis_graph)

glycolysis = subsystem_counts("glycolysis_gluconeogenesis")
print(glycolysis)
assert (
    glycolysis["higher in Th17p"],
    glycolysis["higher in Th17n"],
    glycolysis["measured"],
) == (21, 2, 32)

21 of its 32 measured reactions are red, higher in Th17p, and 2 blue.

### Follow a reaction to its genes

Glycolysis begins with hexokinase, which phosphorylates glucose; Human1 names that reaction MAR04394. In the app a click on it opens its details beside the graph; `element_details` returns them: its activity in each group, its scores, the reaction itself and its genes, each with its own test in this comparison.

In [ ]:
glycolysis_reactions = {
    str(reaction)
    for reaction in model.get_reactions_for_subsystem("glycolysis_gluconeogenesis")
}
hexokinase = element_details(
    dm,
    "reaction",
    "MAR04394",
    color_metric="cohens_d",
    within=glycolysis_reactions,
    dual_evidence=True,
)
facts = {fact["label"]: fact["value"] for fact in hexokinase["facts"]}
main = next(
    direction
    for direction in hexokinase["statistics"]["directions"]
    if direction["main"]
)
print(f"{hexokinase['id']}: {hexokinase['name']}")
print(f"{facts['Equation']} ({facts['Compartment']})")
print(
    f"Cohen's d {main['cohens_d']:.2f}, adjusted p {main['adjusted_pval']:.3f}, significant: {main['significant']}"
)
assert (
    f"{main['cohens_d']:.2f}",
    f"{main['adjusted_pval']:.3f}",
    main["significant"],
) == ("0.16", "0.143", False)

hexokinase_genes = pd.DataFrame(hexokinase["linked"]["rows"]).set_index("label")
assert len(hexokinase_genes) == 5
hexokinase_genes = hexokinase_genes.rename(columns={"scored": "measured"})[
    ["name", "measured", "cohens_d", "adjusted_pval", "significant"]
]
assert sorted(hexokinase_genes.index[~hexokinase_genes["measured"]]) == [
    "GCK",
    "HK3",
    "HKDC1",
]
assert (
    hexokinase_genes.at["HK2", "significant"]
    and hexokinase_genes.at["HK2", "cohens_d"] > 0
)
assert (
    hexokinase_genes.at["HK1", "significant"]
    and hexokinase_genes.at["HK1", "cohens_d"] < 0
)
hexokinase_genes.round(4)

The reaction is not significant (Cohen's d 0.16, adjusted p 0.143), so the graph draws it gray. Of its five genes, *HK2* is higher in Th17p and *HK1* higher in Th17n; *GCK*, *HKDC1* and *HK3* are not measured. In the app a click on *HK2* opens its own details:

In [ ]:
hk2 = element_details(dm, "gene", "HK2", color_metric="cohens_d", dual_evidence=True)
hk2_statistics = hk2["statistics"]["directions"][0]
hk2_percentile = next(
    spread for spread in hk2_statistics["spreads"] if spread["metric"] == "cohens_d"
)
print(
    f"{hk2['label']} ({hk2['name']}): Cohen's d {hk2_statistics['cohens_d']:.3f}, "
    f"adjusted p {hk2_statistics['adjusted_pval']:.4f}, "
    f"percentile {hk2_percentile['percentile']:.0f} of the {hk2_percentile['count']:,} genes' Cohen's d"
)
assert (
    f"{hk2_statistics['cohens_d']:.3f}",
    f"{hk2_statistics['adjusted_pval']:.4f}",
) == ("0.395", "0.0123")
assert f"{hk2_percentile['percentile']:.0f}" == "94"

*HK2* is higher in Th17p (Cohen's d 0.395, adjusted p 0.0123, the 94th percentile of the gene scores), while the reaction its isoforms share does not differ: here gene and reaction disagree.

### Where the TCA cycle changes

The TCA cycle's graph, its view framed on the mitochondrial cycle with `highlight_reactions`; the rest of the subsystem is still drawn around it.

In [ ]:
TCA_STEPS = {
    "aconitase": ["MAR04458", "MAR04589"],
    "isocitrate dehydrogenase": ["MAR03957", "MAR03958", "MAR00710"],
    "oxoglutarate dehydrogenase": ["MAR04209", "MAR06413", "MAR06414"],
    "succinyl-CoA ligase": ["MAR04147", "MAR04152"],
    "succinate dehydrogenase": ["MAR04652"],
    "fumarase": ["MAR04410"],
    "malate dehydrogenase": ["MAR04141"],
    "citrate synthase": ["MAR04145"],
}
tca_id = "tricarboxylic_acid_cycle_and_glyoxylate_dicarboxylate_metabolism"
tca_members = set(members[tca_id])

rows = []
for step, reactions in TCA_STEPS.items():
    for reaction in reactions:
        drawn = reaction if reaction in tca_members else f"{reaction}_back"
        result = reaction_results.loc[drawn]
        rows.append(
            {
                "step": step,
                "reaction": reaction,
                "compartment": model.get_reaction_metadata(reaction)["compartment"],
                "direction drawn": "backward" if drawn.endswith("_back") else "forward",
                "Cohen's d": result["cohens_d"],
                "adjusted p": result["adjusted_pval"],
                "significant": bool(result["is_significant"]),
            }
        )
tca_steps = pd.DataFrame(rows).set_index("reaction")

tca_artifact, tca_graph = draw_pathway(
    "Tricarboxylic acid cycle and glyoxylate/dicarboxylate metabolism"
)
highlight_reactions(
    tca_graph, tca_steps.index[tca_steps["compartment"] == "Mitochondria"], padding=200
)
show(tca_graph)

Each step of the cycle, as Wagner et al.'s Fig. 3A draws it, with the Human1 reactions that carry it, each in the direction the graph draws it:

In [ ]:
higher = {
    "MAR04458": (0.174, 0.018),
    "MAR04589": (0.174, 0.018),
    "MAR04209": (0.246, 0.061),
    "MAR06413": (0.246, 0.061),
    "MAR06414": (0.233, 0.074),
    "MAR04147": (0.148, 0.038),
    "MAR04152": (0.148, 0.038),
}
for reaction, row in tca_steps.iterrows():
    if reaction in higher:
        assert (
            row["significant"]
            and (round(row["Cohen's d"], 3), round(row["adjusted p"], 3))
            == higher[reaction]
        )
    else:
        assert not row["significant"]
assert tca_steps.at["MAR06414", "direction drawn"] == "backward"
assert not reaction_results.loc[["MAR04652", "MAR04652_back"], "is_significant"].any()
red = tca_steps.loc[list(higher), "Cohen's d"]
print(f"Cohen's d of the red steps: {red.min():.2f} to {red.max():.2f}")
assert (f"{red.min():.2f}", f"{red.max():.2f}") == ("0.15", "0.25")
tca_steps.round(3)

In [ ]:
tca = subsystem_counts(tca_id)
print(tca)
assert (
    tca["Delta score"],
    tca["higher in Th17p"],
    tca["higher in Th17n"],
    tca["measured"],
) == (0.333, 12, 2, 30)

tca_results = reaction_results.loc[list(tca_members)]
tca_lower = tca_results[tca_results["is_significant"] & (tca_results["cohens_d"] < 0)]
tca_lower = pd.DataFrame(
    {
        "name": [
            model.get_reaction_metadata(reaction)["name"]
            for reaction in tca_lower.index
        ],
        "compartment": [
            model.get_reaction_metadata(reaction)["compartment"]
            for reaction in tca_lower.index
        ],
        "Cohen's d": tca_lower["cohens_d"],
    },
    index=tca_lower.index,
)
assert sorted(tca_lower["compartment"]) == ["Cytosol", "Peroxisome"]
tca_lower.round(3)

The subsystem's Delta score, +0.33, counts 12 reactions higher in Th17p and 2 lower among its 30; the graph shows where they are. Around the cycle, in the mitochondria, two segments are higher in Th17p, from citrate to isocitrate (aconitase) and from α-ketoglutarate to succinate (oxoglutarate dehydrogenase and succinyl-CoA ligase), broken by isocitrate dehydrogenase and succinate dehydrogenase, as in Wagner et al.'s Fig. 3A. The effects are small, Cohen's d 0.15 to 0.25, so the reds are pale. Human1 groups the cycle with glyoxylate and dicarboxylate metabolism, drawn beside it in the cytosol and the peroxisome, where the subsystem's two blue reactions are.

### Test subsystems with over-representation analysis

The Delta score describes, while over-representation analysis (ORA) tests. It asks whether a subsystem holds more significant reactions in one direction than its share of all measured reactions would predict, and gives each subsystem and direction an enrichment ratio (observed over expected) with an adjusted p-value. The app's settings: Cohen's d decides whether a significant reaction counts as higher or lower, both directions are corrected together with Benjamini-Hochberg, subsystems with fewer than 3 measured reactions are left out, and adjusted p < 0.05 is significant. `run_comparison_set_enrichment` runs it on the comparison, and the workspace installs it.

In [ ]:
reaction_sets = SetEnrichmentFeatureType("reaction")
ora = run_comparison_set_enrichment(
    judged,
    build_set_enrichment_annotations(model, reaction_sets),
    reaction_sets,
    SetEnrichmentConfig(
        direction_metric="cohens_d",
        correction="fdr_bh",
        minimum_measured_set_size=3,
        adjusted_pvalue_threshold=0.05,
        reaction_direction_criterion=dm.reaction_direction_criterion,
    ),
)
dm.install_set_enrichment_artifact(
    dm.capture_analysis_snapshot(analysis_dependencies).provenance, ora
)

ora_table = ora.result_frame()
central_ora = ora_table[ora_table["subsystem_id"].isin(central_carbon)]
assert central_ora.loc[central_ora["is_significant"], "subsystem_name"].tolist() == [
    "Glycolysis / Gluconeogenesis"
]
higher_ora = central_ora[central_ora["direction"] == "target_higher"].set_index(
    "subsystem_name"
)
glycolysis_ora = higher_ora.loc["Glycolysis / Gluconeogenesis"]
oxphos_ora = higher_ora.loc["Oxidative phosphorylation"]
assert (glycolysis_ora["overlap_size"], glycolysis_ora["subsystem_size"]) == (21, 32)
assert f"{glycolysis_ora['adjusted_pval']:.3f}" == "0.010"
assert (oxphos_ora["overlap_size"], oxphos_ora["subsystem_size"]) == (6, 9)
assert f"{oxphos_ora['adjusted_pval']:.2f}" == "0.32"
higher_ora[
    [
        "overlap_size",
        "subsystem_size",
        "enrichment_ratio",
        "adjusted_pval",
        "is_significant",
    ]
].round(3)

The network colored by ORA, each subsystem at its stronger direction as a signed log2(enrichment ratio), and `plot_set_enrichment_category`, its ranking: red when enriched among the reactions higher in Th17p and blue among the lower ones. `GraphCalculationInput.from_workspace` presents the workspace to the pipeline, and `prepare_set_enrichment_overview` gives it the ORA's colors.

In [ ]:
ora_source = GraphCalculationInput.from_workspace(dm)
ora_source.prepare_set_enrichment_overview(ora)
show(
    draw_network(
        "cohens_d",
        source=ora_source,
        visualization_mode="group_comparison",
        set_enrichment_artifact=ora,
    )
)
show(plot_set_enrichment_category(ora, category_numbers=numbers))

A subsystem significant here holds more changed reactions than chance would give it. Of Central Carbon & Bioenergetics, only Glycolysis / Gluconeogenesis is, with 21 of its 32 measured reactions higher in Th17p (adjusted p 0.010). Oxidative phosphorylation, with the category's strongest Delta score, is not: 6 of its 9 reactions higher, adjusted p 0.32.

### Check a published finding

Wagner et al., whose data this is, found with Compass that the polyamine pathway was among the most associated with Th17 pathogenicity, Th17n being more active in the arginine pathways upstream of putrescine (Figs. 4A, 4B), and measured *SAT1* higher in Th17p (Fig. 4D). Human1 places polyamine synthesis and turnover in Arginine and proline metabolism.

In [ ]:
arginine_artifact, arginine_graph = draw_pathway("Arginine and proline metabolism")
show(arginine_graph)

sat1_result = gene_results.loc[sat1]
arginine_delta = subsystem_scores.at["arginine_and_proline_metabolism", "delta"]
print(
    f"SAT1: Cohen's d {sat1_result['cohens_d']:.2f}, adjusted p {sat1_result['adjusted_pval']:.1e}"
)
print(
    f"Arginine and proline metabolism: {arginine_counts['significant']} of {arginine_counts['measured']} "
    f"measured reactions differ, Delta score {arginine_delta:.2f}"
)
assert f"{sat1_result['cohens_d']:.2f}" == "1.48" and sat1_result["is_significant"]
assert f"{arginine_delta:.2f}" == "0.08"

*SAT1* is red: higher in Th17p (Cohen's d 1.48), as Wagner et al. measured it. The subsystem's reactions move both ways: 34 of its 49 measured reactions differ, in either direction, so its Delta score is near zero (0.08).

A metabolite's details list the reactions that make and use it. Those of putrescine, in the cytosol, within this subsystem:

In [ ]:
arginine_reactions = {
    str(reaction)
    for reaction in model.get_reactions_for_subsystem("arginine_and_proline_metabolism")
}
putrescine = element_details(
    dm, "metabolite", "MAM02812c", color_metric="cohens_d", within=arginine_reactions
)
print(f"{putrescine['label']} ({putrescine['id']})")
pd.DataFrame(putrescine["linked"]["rows"]).set_index("label").rename(
    columns={"scored": "measured"}
)[["name", "role", "measured", "cohens_d", "adjusted_pval", "significant"]].round(3)

Under a subsystem's graph the app lists the Metabolic Atlas maps that draw its reactions, and opens each with the subsystem in view. `maps_for_reactions` lists them, the most reactions first. The first is drawn below, `highlight_reactions` framing the subsystem's reactions and `add_map_bands` naming its band.

In [ ]:
arginine_maps = maps_for_reactions(model, arginine_reactions)
for entry, count in arginine_maps:
    print(f"{entry.name}: {count} reaction{'s' if count != 1 else ''}")

first_map = arginine_maps[0][0]
_, map_plot = draw_pathway(first_map.map_id, graph_type="compartment", hover="light")
highlight_reactions(
    map_plot, map_ids_for_reactions(model, first_map.map_id, arginine_reactions)
)
bands = tuple(sorted(map_bands(first_map.map_id), key=str.lower))
add_map_bands(
    map_plot, first_map.map_id, band_name(bands, "Arginine and proline metabolism")
)
map_plot.title.text = f"{first_map.name} map, Arginine and proline metabolism in view"
show(map_plot)

### One group's activity

Single-group activity colors the pathways by one group's own activity, with nothing compared, so there are no p-values or Delta scores. `activate_activity_mode` scores one group, here Th17p by the mean of its samples, and makes it what the graphs are colored by. The network colors each subsystem by the activity of its measured reactions, from purple (low) through light gray to gold (high), and `plot_subsystem_activity_ranking` gives each subsystem's activity score beside each reaction's activity as a dot.

In [ ]:
activity_config = ActivityConfig(
    group_id="Th17p", summary=GroupSummary("mean"), minimum_subsystem_coverage=0.0
)
dm.activate_activity_mode(activity_config)
activity = dm.get_activity_artifact()

show(
    draw_network(
        activity_config.metric_key_for(ActivityOverlay.REACTION),
        visualization_mode="activity",
        activity_artifact=activity,
        activity_overlay=ActivityOverlay.REACTION,
    )
)
show(
    plot_subsystem_activity_ranking(
        dm,
        activity,
        overlay=ActivityOverlay.REACTION,
        max_points_per_subsystem=50,
        return_plot=True,
    )
)

The colors are relative within the feature family, so gold marks the pathways most active in Th17p and purple the least, compared with its other pathways, not with Th17n.

## 5. Adjust for depth

This section checks whether the result of section 3 holds with a technical difference between the groups, sequencing depth, in the model. The sheet's `NALIGNED` column holds each sample's aligned reads.

In [ ]:
sheet = dm.get_grouping_metadata().table
aligned_reads = pd.to_numeric(sheet["NALIGNED"]).groupby(sheet["cell_type"]).median()
depth_ratio = aligned_reads["Th17p"] / aligned_reads["Th17n"]
print(
    f"Median aligned reads: Th17n {aligned_reads['Th17n']:,.0f}, Th17p {aligned_reads['Th17p']:,.0f} ({depth_ratio:.2f} times)"
)
assert f"{depth_ratio:.2f}" == "0.86"

Th17p cells were sequenced 0.86 times as deep as Th17n cells (median aligned reads). The moderated t-test (limma) can compare the groups at equal depth: `ComparisonAdjustment.from_metadata` reads `NALIGNED` off the sheet, on a log10 scale, and the test adjusts for it. `with_thresholds` judges the adjusted comparison at the same adjusted p of 0.1.

In [ ]:
adjustment = ComparisonAdjustment.from_metadata(
    sheet, (AdjustmentColumn("NALIGNED", True),)
)
adjusted = GroupComparison(
    spec=comparison_spec, test="moderated_t", correction="fdr_bh", adjustment=adjustment
)
for for_genes in (False, True):
    adjusted.run_test(dm, for_genes=for_genes, test="moderated_t", correction="fdr_bh")

adjusted_snapshot = dm.capture_analysis_snapshot(analysis_dependencies)
at_equal_depth = with_thresholds(
    adjusted.build_artifact(
        provenance=adjusted_snapshot.provenance,
        source_fingerprints=dict(dm.source_fingerprints),
    ),
    {"pval": 0.1, "cohens_d": None, "log2_fc": None, "mean_difference": None},
    {"pval": 0.1, "cohens_d": None, "log2_fc": None, "mean_difference": None},
)
sat1_adjusted = at_equal_depth.get_test_results(for_genes=True).loc[sat1]
print(
    f"Reactions significant at adjusted p < 0.1: {len(at_equal_depth.significant_reactions):,} at equal depth, "
    f"{len(significant_reactions):,} with the rank-sum test"
)
print(
    f"SAT1: Cohen's d {sat1_adjusted['cohens_d']:.2f} at equal depth "
    f"(significant: {bool(sat1_adjusted['is_significant'])}), {sat1_result['cohens_d']:.2f} with the rank-sum test"
)
assert len(at_equal_depth.significant_reactions) == 2804
assert f"{sat1_adjusted['cohens_d']:.2f}" == "1.49" and sat1_adjusted["is_significant"]

At equal depth 2,804 reactions pass adjusted p < 0.1, against 3,039 with the rank-sum test. *SAT1* keeps its direction and significance with depth in the model (Cohen's d 1.49, against 1.48).